# 1. Imports necessários

In [37]:
%pip -q install cloudscraper bs4 pandas nltk spacy 

import cloudscraper
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
import subprocess
import sys
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Scraping dos dados

In [ ]:
def realizar_scraping():
    scraper = cloudscraper.create_scraper(
        browser={
            'browser': 'chrome',
            'platform': 'windows',
            'desktop': True
        }
    )
    
    base_url = "https://backloggd.com"
    links_dos_jogos = []
    pagina_atual = 1

    print("--- ETAPA 1: Coletando links dos jogos ---")
    
    limite_jogos = 200
    
    while len(links_dos_jogos) < limite_jogos:
        if pagina_atual > 10:
            print("Passou de 10 páginas e não achou tudo. Parando por segurança.")
            break

        url_paginada = f"{base_url}/games/lib/popular/?page={pagina_atual}"
        print(f"Acessando página {pagina_atual} de populares...")
        
        response = scraper.get(url_paginada)
        
        if response.status_code != 200:
            print(f"Erro ao acessar! Status: {response.status_code}.")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        links_html = soup.find_all("a", class_="cover-link")
        
        if not links_html:
            print("Nenhum link encontrado! O site pode ter bloqueado ou mudado o HTML.")
            break
            
        for link in links_html:
            caminho = link.get("href")
            if caminho and caminho not in links_dos_jogos:
                links_dos_jogos.append(caminho)
            
            if len(links_dos_jogos) == limite_jogos:
                break
                
        pagina_atual += 1
        time.sleep(random.uniform(2.5, 4.5))

    if links_dos_jogos:
        print(f"\nSucesso! {len(links_dos_jogos)} links coletados.")
        print("\n--- ETAPA 2: Extraindo dados de cada jogo ---")
        
        dados_finais = []

        for index, caminho in enumerate(links_dos_jogos, start=1):
            url_jogo = base_url + caminho
            print(f"[{index}/{limite_jogos}] Extraindo dados de: {url_jogo}")
            
            res = scraper.get(url_jogo)
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")
                
                # 1. NOME
                nome_tag = soup.find("h1")
                nome = nome_tag.get_text(strip=True) if nome_tag else "N/A"
                
                # 2. DESCRIÇÃO
                descricao = "N/A"
                div_summary = soup.find("div", id="collapseSummary")
                if div_summary:
                    p_tag = div_summary.find("p", class_="mb-0")
                    # Pegamos o texto usando espaço como separador
                    desc_bruta = p_tag.get_text(separator=" ", strip=True) if p_tag else div_summary.get_text(separator=" ", strip=True)
                    # A mágica para o Excel: Troca enter/quebra de linha (\n) e múltiplos espaços por 1 espaço simples!
                    descricao = re.sub(r'\s+', ' ', desc_bruta)
                
                # 3. TAGS
                tags_html = soup.find_all("a", class_="game-details-value")
                tags_brutas = [tag.get_text(strip=True) for tag in tags_html if "genre" in tag.get("href", "")]
                tags = list(dict.fromkeys(tags_brutas)) # Remove duplicatas (PC vs Mobile)

                # 4. DATA DE LANÇAMENTO
                # Busca qualquer link que contenha 'release_year' no href
                data_tag = soup.find("a", href=re.compile(r"release_year"))
                data_lancamento = data_tag.get_text(strip=True) if data_tag else "N/A"

                # 5. DESENVOLVEDORAS / PUBLISHERS
                empresas_tags = soup.find_all("a", href=re.compile(r"/company/"))
                # Pega os nomes e remove as duplicatas (PC vs Mobile) mantendo a ordem
                empresas = list(dict.fromkeys([emp.get_text(strip=True) for emp in empresas_tags]))

                # 6. PLATAFORMAS
                plataformas_tags = soup.find_all("a", class_="game-page-platform")
                # Remove duplicatas
                plataformas = list(dict.fromkeys([p.get_text(strip=True) for p in plataformas_tags]))

                # 7. NOTA MÉDIA
                nota = "N/A"
                rating_div = soup.find("div", id="game-rating")
                if rating_div:
                    h1 = rating_div.find("h1")
                    if h1:
                        nota = h1.get_text(strip=True)

                # 8. TEMPOS DE JOGO
                tempos_brutos = []
                for tp in soup.find_all("div", class_="time-played"):
                    val = tp.find(class_="element-revealed")
                    lbl = tp.find(class_="label")
                    if val and lbl:
                        tempos_brutos.append(f"{lbl.get_text(strip=True)}: {val.get_text(strip=True)}")
                tempos_lista = list(dict.fromkeys(tempos_brutos))

                # 9. REVIEWS E LIKES
                qtd_reviews = "N/A"
                qtd_likes = "N/A"
                
                # Varre os blocos onde esses números costumam ficar e checa pelo texto (Reviews ou Likes)
                for container in soup.find_all("div", class_="center-container"):
                    p_tag = container.find("p")
                    h3_tag = container.find("h3")
                    if p_tag and h3_tag:
                        texto_p = p_tag.get_text(strip=True).lower()
                        if "reviews" in texto_p:
                            qtd_reviews = h3_tag.get_text(strip=True)
                        elif "likes" in texto_p:
                            qtd_likes = h3_tag.get_text(strip=True)

                # SALVANDO OS DADOS
                dados_finais.append({
                    "Posicao": index,
                    "Nome": nome,
                    "Lançamento": data_lancamento,
                    "Desenvolvedora/Publisher": ", ".join(empresas) if empresas else "N/A",
                    "Plataformas": ", ".join(plataformas) if plataformas else "N/A",
                    "Nota Média": nota,
                    "Tempo de Jogo": " | ".join(tempos_lista) if tempos_lista else "N/A",
                    "Qtd Reviews": qtd_reviews,
                    "Qtd Likes": qtd_likes,
                    "Tags": ", ".join(tags) if tags else "N/A",
                    "Descricao": descricao
                })
                
            else:
                print(f"Falha ao carregar o jogo. Erro {res.status_code}")
                
            time.sleep(random.uniform(3.0, 5.0)) # Tempo seguro para evitar block

        if dados_finais:
            print("\n--- RESULTADO FINAL ---")
            df = pd.DataFrame(dados_finais)
            print(df)
            df.to_csv("top_jogos_backloggd.csv", index=False, encoding='utf-8-sig') # utf-8-sig evita erro com caracteres no Excel

realizar_scraping()

# 3. Preparação do ambiente 

In [39]:
for recurso in ["punkt", "punkt_tab", "stopwords", "rslp"]:
    nltk.download(recurso, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
        check=True,
    )
    nlp = spacy.load("en_core_web_sm")


try:
    df_jogos = pd.read_csv("top_jogos_backloggd.csv")
    print(f"Base de dados carregada! {len(df_jogos)} jogos encontrados.\n")
except FileNotFoundError:
    print("Erro: O arquivo 'top_jogos_backloggd.csv' não foi encontrado.")
    sys.exit()

frases = df_jogos.to_dict('records')

Base de dados carregada! 199 jogos encontrados.



# 4. Tokenização

In [40]:
tokens = [
    word_tokenize(str(jogo["Descricao"]), language="english")
    for jogo in frases
]

for jogo, tokens_do_jogo in zip(frases, tokens):
    print(f"Jogo: {jogo['Nome']}")
    print(f"Texto bruto: {jogo['Descricao']}")
    print(f"Tokens:      {tokens_do_jogo}\n")
    print("-" * 60)

Jogo: Grand Theft Auto V
Texto bruto: Grand Theft Auto V is a vast open world game set in Los Santos, a sprawling sun-soaked metropolis struggling to stay afloat in an era of economic uncertainty and cheap reality TV. The game blends storytelling and gameplay in new ways as players repeatedly jump in and out of the lives of the game’s three lead characters, playing all sides of the game’s interwoven story.
Tokens:      ['Grand', 'Theft', 'Auto', 'V', 'is', 'a', 'vast', 'open', 'world', 'game', 'set', 'in', 'Los', 'Santos', ',', 'a', 'sprawling', 'sun-soaked', 'metropolis', 'struggling', 'to', 'stay', 'afloat', 'in', 'an', 'era', 'of', 'economic', 'uncertainty', 'and', 'cheap', 'reality', 'TV', '.', 'The', 'game', 'blends', 'storytelling', 'and', 'gameplay', 'in', 'new', 'ways', 'as', 'players', 'repeatedly', 'jump', 'in', 'and', 'out', 'of', 'the', 'lives', 'of', 'the', 'game', '’', 's', 'three', 'lead', 'characters', ',', 'playing', 'all', 'sides', 'of', 'the', 'game', '’', 's', 'inte

# 5. Normalização

In [41]:
def normalizar_tokens(tokens_da_frase):
    return [
        token.casefold()
        for token in tokens_da_frase
        if token.isalpha()
    ]

tokens_normalizados = [normalizar_tokens(lista) for lista in tokens]

for jogo, lista_original, lista_normalizada in zip(frases, tokens, tokens_normalizados):
    print(f"Jogo:         {jogo['Nome']}")
    print(f"Texto bruto:  {jogo['Descricao']}")
    print(f"Tokens:       {lista_original}")
    print(f"Normalizados: {lista_normalizada}\n")
    print("-" * 80)

Jogo:         Grand Theft Auto V
Texto bruto:  Grand Theft Auto V is a vast open world game set in Los Santos, a sprawling sun-soaked metropolis struggling to stay afloat in an era of economic uncertainty and cheap reality TV. The game blends storytelling and gameplay in new ways as players repeatedly jump in and out of the lives of the game’s three lead characters, playing all sides of the game’s interwoven story.
Tokens:       ['Grand', 'Theft', 'Auto', 'V', 'is', 'a', 'vast', 'open', 'world', 'game', 'set', 'in', 'Los', 'Santos', ',', 'a', 'sprawling', 'sun-soaked', 'metropolis', 'struggling', 'to', 'stay', 'afloat', 'in', 'an', 'era', 'of', 'economic', 'uncertainty', 'and', 'cheap', 'reality', 'TV', '.', 'The', 'game', 'blends', 'storytelling', 'and', 'gameplay', 'in', 'new', 'ways', 'as', 'players', 'repeatedly', 'jump', 'in', 'and', 'out', 'of', 'the', 'lives', 'of', 'the', 'game', '’', 's', 'three', 'lead', 'characters', ',', 'playing', 'all', 'sides', 'of', 'the', 'game', '’', 

# 6. Remoção de stopwords

In [42]:
stopwords_en = set(stopwords.words("english"))

def remover_stopwords(tokens_da_frase):
    return [token for token in tokens_da_frase if token not in stopwords_en]

tokens_sem_stopwords = [
    remover_stopwords(lista) for lista in tokens_normalizados
]

for jogo, lista_normalizada, lista_filtrada in zip(frases, tokens_normalizados, tokens_sem_stopwords):
    print(f"Jogo:              {jogo['Nome']}")
    print(f"Texto bruto:       {jogo['Descricao']}")
    print(f"Normalizados:      {lista_normalizada}")
    print(f"Sem stopwords:     {lista_filtrada}\n")
    print("-" * 80)

Jogo:              Grand Theft Auto V
Texto bruto:       Grand Theft Auto V is a vast open world game set in Los Santos, a sprawling sun-soaked metropolis struggling to stay afloat in an era of economic uncertainty and cheap reality TV. The game blends storytelling and gameplay in new ways as players repeatedly jump in and out of the lives of the game’s three lead characters, playing all sides of the game’s interwoven story.
Normalizados:      ['grand', 'theft', 'auto', 'v', 'is', 'a', 'vast', 'open', 'world', 'game', 'set', 'in', 'los', 'santos', 'a', 'sprawling', 'metropolis', 'struggling', 'to', 'stay', 'afloat', 'in', 'an', 'era', 'of', 'economic', 'uncertainty', 'and', 'cheap', 'reality', 'tv', 'the', 'game', 'blends', 'storytelling', 'and', 'gameplay', 'in', 'new', 'ways', 'as', 'players', 'repeatedly', 'jump', 'in', 'and', 'out', 'of', 'the', 'lives', 'of', 'the', 'game', 's', 'three', 'lead', 'characters', 'playing', 'all', 'sides', 'of', 'the', 'game', 's', 'interwoven', 'stor

# 7. Lematização

In [43]:
def lematizar(tokens_da_frase):
    texto_processavel = " ".join(tokens_da_frase)
    documento = nlp(texto_processavel)
    return [token.lemma_ for token in documento]

lemas = [lematizar(lista) for lista in tokens_sem_stopwords]

for jogo, lista_filtrada, lemas_da_frase in zip(frases, tokens_sem_stopwords, lemas):
    print(f"Jogo:          {jogo['Nome']}")
    print(f"Sem stopwords: {lista_filtrada}")
    print(f"Lemas:         {lemas_da_frase}\n")
    print("-" * 80)

Jogo:          Grand Theft Auto V
Sem stopwords: ['grand', 'theft', 'auto', 'v', 'vast', 'open', 'world', 'game', 'set', 'los', 'santos', 'sprawling', 'metropolis', 'struggling', 'stay', 'afloat', 'era', 'economic', 'uncertainty', 'cheap', 'reality', 'tv', 'game', 'blends', 'storytelling', 'gameplay', 'new', 'ways', 'players', 'repeatedly', 'jump', 'lives', 'game', 'three', 'lead', 'characters', 'playing', 'sides', 'game', 'interwoven', 'story']
Lemas:         ['grand', 'theft', 'auto', 'v', 'vast', 'open', 'world', 'game', 'set', 'los', 'santos', 'sprawl', 'metropolis', 'struggle', 'stay', 'afloat', 'era', 'economic', 'uncertainty', 'cheap', 'reality', 'tv', 'game', 'blend', 'storytelle', 'gameplay', 'new', 'way', 'player', 'repeatedly', 'jump', 'life', 'game', 'three', 'lead', 'character', 'play', 'side', 'game', 'interweave', 'story']

--------------------------------------------------------------------------------
Jogo:          Red Dead Redemption 2
Sem stopwords: ['red', 'dead', 

# 8. Stemming

In [44]:
stemmer = PorterStemmer()

def aplicar_stemming(tokens_da_frase):
    return [stemmer.stem(token) for token in tokens_da_frase]

stems = [aplicar_stemming(lista) for lista in tokens_sem_stopwords]

for jogo, lista_filtrada, stems_da_frase in zip(frases, tokens_sem_stopwords, stems):
    print(f"Jogo:          {jogo['Nome']}")
    print(f"Sem stopwords: {lista_filtrada}")
    print(f"Stems:         {stems_da_frase}\n")
    print("-" * 80)

Jogo:          Grand Theft Auto V
Sem stopwords: ['grand', 'theft', 'auto', 'v', 'vast', 'open', 'world', 'game', 'set', 'los', 'santos', 'sprawling', 'metropolis', 'struggling', 'stay', 'afloat', 'era', 'economic', 'uncertainty', 'cheap', 'reality', 'tv', 'game', 'blends', 'storytelling', 'gameplay', 'new', 'ways', 'players', 'repeatedly', 'jump', 'lives', 'game', 'three', 'lead', 'characters', 'playing', 'sides', 'game', 'interwoven', 'story']
Stems:         ['grand', 'theft', 'auto', 'v', 'vast', 'open', 'world', 'game', 'set', 'lo', 'santo', 'sprawl', 'metropoli', 'struggl', 'stay', 'afloat', 'era', 'econom', 'uncertainti', 'cheap', 'realiti', 'tv', 'game', 'blend', 'storytel', 'gameplay', 'new', 'way', 'player', 'repeatedli', 'jump', 'live', 'game', 'three', 'lead', 'charact', 'play', 'side', 'game', 'interwoven', 'stori']

--------------------------------------------------------------------------------
Jogo:          Red Dead Redemption 2
Sem stopwords: ['red', 'dead', 'redemptio

# 9. Pipeline completa

In [45]:
def processar_texto(texto):
    tokens_locais = word_tokenize(str(texto), language="english")
    normalizados_locais = normalizar_tokens(tokens_locais)
    sem_stopwords_locais = remover_stopwords(normalizados_locais)

    return pd.Series({
        "tokens": tokens_locais,
        "tokens_normalizados": normalizados_locais,
        "tokens_sem_stopwords": sem_stopwords_locais,
        "lemas": lematizar(sem_stopwords_locais),
        "stems": aplicar_stemming(sem_stopwords_locais),
    })

print("Processando os textos. Isso pode levar alguns segundos dependendo da quantidade de jogos...\n")

colunas_processadas = df_jogos["Descricao"].apply(processar_texto)
df_processado = pd.concat([df_jogos, colunas_processadas], axis=1)

pd.set_option("display.max_colwidth", 100) # Deixa a coluna mais larga para lermos as listas
pd.set_option("display.max_columns", None) # Garante que o Pandas não esconda colunas
pd.set_option("display.width", 1000)

print(df_processado[['Nome', 'tokens_normalizados', 'tokens_sem_stopwords', 'lemas', 'stems']].head(3))

df_processado.to_csv("jogos_nlp_completo.csv", index=False, encoding='utf-8-sig')
print("\nProcessamento concluído e salvo no arquivo 'jogos_nlp_completo.csv'!")

Processando os textos. Isso pode levar alguns segundos dependendo da quantidade de jogos...

                    Nome                                                                                  tokens_normalizados                                                                                 tokens_sem_stopwords                                                                                                lemas                                                                                                stems
0     Grand Theft Auto V  [grand, theft, auto, v, is, a, vast, open, world, game, set, in, los, santos, a, sprawling, metr...  [grand, theft, auto, v, vast, open, world, game, set, los, santos, sprawling, metropolis, strugg...  [grand, theft, auto, v, vast, open, world, game, set, los, santos, sprawl, metropolis, struggle,...  [grand, theft, auto, v, vast, open, world, game, set, lo, santo, sprawl, metropoli, struggl, sta...
1  Red Dead Redemption 2  [red, dead, redemption,